# 00 — 理論理解: GRF · MPC · WBC（MPC設計者向け）

**対象:** 四足制御初心者 + ADAS操舵MPC経験者  
**ゴール:** 「何を調整すると何が起きるか」を **失敗と成功のパターン** として理解する

---

## この Notebook の進め方

1. **Step 1–4:** 3層パイプラインと数式を「直觉」で理解  
2. **Step 5–6:** 摩擦円錐を **目で見る**（μ の意味）  
3. **Step 7:** MPC設計者向け **調整マトリクス** を読む  
4. **Step 8:** 簡単な数値実験（SRB の合力）  
5. **Step 9:** デモ Notebook へ進む

> 詳細版: [WORKSHOP.md](../WORKSHOP.md)


In [ ]:
import sys
from pathlib import Path

# mpc_dog ルートを sys.path に追加
ROOT = Path.cwd()
for p in [ROOT, *ROOT.parents]:
    if (p / "scripts" / "pympc_lab.py").exists():
        ROOT = p
        break
sys.path.insert(0, str(ROOT / "scripts"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from pympc_lab import (
    TUNING_GUIDE,
    apply_preset,
    compare_runs,
    load_param_study,
    load_preset_yaml,
    plot_friction_cone,
    run_flat_sim,
    run_speed_terrain_sim,
    run_speed_terrain_sim_resilient,
)

%matplotlib inline
plt.rcParams["figure.figsize"] = (9, 4)
print(f"repo: {ROOT}")


## Step 1 — 四足制御の「定番3層」（対立ではなく接続）

```
速度指令 / ゲイト
      ↓
MPC (SRB)  … 最適化変数 = 接地反力 GRF（12次元）
      ↓
WBC相当     … Stance: GRF→関節τ / Swing: 足軌道+PD
      ↓
MuJoCo / 実機
```

**ADAS MPC との対応**

| 操舵MPC | 四足PyMPC |
|---------|-----------|
| 車両モデル | SRB（箱1個） |
| 横G・舵角制約 | **摩擦円錐** |
| MPC出力 | **GRF**（タイヤ力相当） |
| 下位 | WBC → 関節トルク |

**初心者向け一言:** MPCは「各足が地面を **どれだけ蹴るか**」を決める。WBCは「その蹴りを **関節で実現** する」。


## Step 2 — GRF（Ground Reaction Force）とは

- 足先と地面の間の力 $\mathbf{F}_i = (F_{ix}, F_{iy}, F_{iz})$
- 4足 → **12次元** の入力（低次元で物理制約を入れやすい）
- ロボットの加速は $\sum_i \mathbf{F}_i$（+ 重力）で決まる

**なぜ関節角度を直接MPCしないのか？**
- 次元が高い（12関節以上）
- 摩擦円錐など **接触力の物理** を入れにくい
- GRFを決めれば CoM 運動を計画できる → 関節はWBCに任せる


## Step 3 — MPC の数式（口頭説明用）

離散時間ホライゾン $N$、サンプリング $\Delta t$:

$$\min \sum_{k=0}^{N-1} \|x_k - x_k^{ref}\|_Q + \|u_k\|_R
\quad \text{s.t.} \quad x_{k+1} = f_{SRB}(x_k, u_k)$$

**摩擦円錐**（各足 $i$、接触中）:

$$\sqrt{F_{ix}^2 + F_{iy}^2} \le \mu F_{iz}, \quad F_{iz}^{min} \le F_{iz} \le F_{iz}^{max}$$

**Convex化のコツ:** どの足が stance/swing かを **ゲイトで固定** → GRFについて凸に近づける。

PyMPC デフォルト: $N=12$, $\Delta t=0.02$ s → **0.24 s 先読み**


## Step 4 — WBC 相当層

| 脚 | 処理 |
|----|------|
| **Stance** | MPCの $\mathbf{F}_i$ → $\boldsymbol{\tau} = \mathbf{J}^\top \mathbf{F} + \text{PD}$ |
| **Swing** | Bezier足軌道 + PD（MPCのGRFは使わない） |

MPC設計者が触るのは主に **MPC層**。WBCゲインは「追従の硬さ」= 計画通りに蹴れるかどうか。


## Step 5 — 摩擦円錐を可視化（μ を上げ下げすると？）

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, mu in zip(axes, [0.3, 0.5, 0.8]):
    plot_friction_cone(mu=mu, f_max=120, ax=ax)
fig.suptitle("Higher mu allows larger horizontal Fx for the same Fz", y=1.02)
plt.tight_layout()


**MPC設計者メモ**

| μ | 典型の使いどころ | 失敗パターン |
|---|----------------|--------------|
| 低 (0.35–0.45) | 氷・濡れ床・不整地 | 加速不足、指令追従が鈍い |
| 中 (0.5) | 平坦デフォルト | — |
| 高 (0.6+) | 高摩擦床・積極走行 | 転倒・横滑り・オーバーシュート |

⚠️ **sim の地面摩擦** と **MPC の μ** は別パラメータ。両方の意味を混同しないこと。


## Step 6 — SRB の合力デモ（F=ma の直觉）

In [ ]:
m = 15.0  # Go2 近似 [kg]
g = 9.81
# 4足がそれぞれ垂直100N、前足2本が追加で水平30N前向き
Fz_total = 4 * 100
Fx_total = 2 * 30
ax = Fx_total / m
print(f"合力 Fx={Fx_total}N → ax={ax:.2f} m/s²")
print(f"垂直 Fz={Fz_total}N vs mg={m*g:.0f}N → 浮き/沈み: {(Fz_total - m*g):+.0f}N")

# mu=0.5 の摩擦円錐内か？
mu = 0.5
for name, fx, fz in [("前足FL", 30, 100), ("過剰水平", 80, 100)]:
    ok = abs(fx) <= mu * fz
    print(f"{name}: |Fx|={abs(fx)} <= mu*Fz={mu*fz:.0f} ? {ok}")


## Step 7 — MPC設計者向け 調整マトリクス（成功/失敗パターン）

In [ ]:
df = pd.DataFrame(TUNING_GUIDE)
cols = ["param", "what", "raise", "lower", "failure_symptom", "failure_fix", "success_sign"]
display(df[cols])


### 現場で使うトリアージ（転倒したら）

1. **即転倒（数秒以内）** → `ref_z`↑, `step_freq`↓, `mu`↓, 足場opt OFF  
2. **加速しない** → `mu`↑（ただし転倒リスク）, `grf_max`↑, 速度指令↓  
3. **滑る・横倒れ** → `mu`↓, `duty_factor`↑  
4. **不整地で変な足場** → 足場opt OFF で比較 → 地形推定を疑う  
5. **MPCが遅い** → `solver_mode='speed'`, horizon↓

次の Notebook で **実際に sim を回して** 体感します。


## Step 8 — チェックリスト

- [ ] GRF / MPC / WBC の役割を1文ずつ説明できる  
- [ ] 摩擦円錐が何を制約しているか説明できる  
- [ ] `mu` を上げると **何が起きやすいか**（加速 vs 転倒）を説明できる  
- [ ] 転倒時の最初の3つのアクションを言える  

---

## 次へ

| Notebook | 内容 |
|----------|------|
| [01_demo_session01_flat_smoke.ipynb](./01_demo_session01_flat_smoke.ipynb) | 平坦スモーク + GRF可視化 |
| [02_demo_session02_flat_tune.ipynb](./02_demo_session02_flat_tune.ipynb) | μ / 歩調チューニング |
| [03_demo_session03a_rough_boxes.ipynb](./03_demo_session03a_rough_boxes.ipynb) | 不整地 boxes |
| [04_demo_session03b_rough_perlin.ipynb](./04_demo_session03b_rough_perlin.ipynb) | 不整地 perlin |
